# History from nsepy

In [ ]:
# hack to change jupyter directory in notebooks for imports
import os
from pathlib import Path
if Path.cwd().parts[-1] == 'notebooks':
    root = Path.cwd().parent
else:
    root = Path.cwd()
os.chdir(root)

# Logger
import logging, logging.config
import yaml

with open(root / 'config' / 'log.yml', 'r') as f:
    config = yaml.safe_load(f.read())
    logging.config.dictConfig(config)

log = logging.getLogger('ib_log')

In [14]:
from nsepy import get_history
from datetime import datetime, timedelta, time
import pandas as pd
import tqdm

#inputs
symbol = 'MIDCPNIFTY'
is_index=True
days = 365

end = datetime.now().date()
start = end-timedelta(days=days)
cols = {'Date': 'date',
        'Open': 'open',
        'High': 'high',
        'Low': 'low',
        'Close': 'close',
        'Volume': 'volume',
        'Turnover': 'turnover',
        'Symbol': 'symbol',
        'Series': 'series',
        'Prev Close': 'prev_cls',
        'Last': 'last',
        'VWAP': 'vwap',
        'Trades': 'trades',
        'Deliverable Volume': 'd_volume',
        '%Deliverble': 'pct_delv'}

# call
data = get_history(symbol=symbol, start=start, end=end, index=is_index).reset_index()
if data.empty:
        log.error(f'Could not generate history for {symbol}')
        data = None 
        # return

data.insert(0, 'date', data.Date.apply(pd.to_datetime)) 
data.drop('Date', axis=1, inplace=True) # remove old Date field

# convert to end of day India time
date=(data.date + pd.Timedelta(hours=16)).dt.tz_localize('Asia/Calcutta')
data = data.assign(date=date)
# if index, put in symbol column
if is_index:
        data.insert(0, 'Symbol', symbol)

data.rename(columns=cols, inplace=True)

2023-03-15 08:39:28,189 | ib_log | Could not generate history for MIDCPNIFTY


AttributeError: 'NoneType' object has no attribute 'insert'

In [11]:
data.empty

True

In [ ]:
# get the symbols
from src.nse.nse import get_nse_syms
df_syms = get_nse_syms()

# convert to scrips with is_index = True for Indexes like NIFTY
scrips = df_syms.set_index('symbol')['secType'].eq('IND').to_dict()

In [ ]:
# Gets history if they are old

from pathlib import Path
import pandas as pd
from tqdm.notebook import tqdm_notebook

from src.nse.nse import get_nse_hist
from src.support import pickle_age

In [ ]:
days_old = 1 # set how many days old
hists_file = root / 'data' / 'master'

try:
    hist_age_delta = pickle_age(hists_file)['nse_hists.pkl']
    hist_age = hist_age_delta.days + hist_age_delta.hours/24 + hist_age_delta.minutes/24/60
except KeyError:
    hist_age = days_old + 1 # regenerate

old_hists = hist_age > days_old

dfs = []
hist_days = 365

if old_hists:
    tq_scrips = tqdm_notebook(scrips.items(), bar_format="{desc:<8}{percentage:3.0f}%{r_bar}")
    for k, v in tq_scrips:
        try:
            dfs.append(get_nse_hist(k, v, hist_days))
        except KeyError as e:
            log.error(f"{k} has error {e}")
            pass
        tq_scrips.set_description(f"{k}")
else:
    dfs = [pd.read_pickle(hists_file / 'nse_hists.pkl')]

# assemble the hists
nse_hists = pd.concat(dfs,ignore_index=True)

# store the nse_hists if new
if old_hists:
    data_path = root / 'data' / 'master' / 'nse_hists.pkl'
    nse_hists.to_pickle(data_path)